In [1]:
import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RANDOM_STATE = 42

# Concatenação e split temporal (70/30)

Este notebook:
1) Carrega os CSVs por classe (um arquivo `*_features_processed_timestamp.csv` por pasta).
2) Normaliza nomes de colunas (ex.: `Label` → `label`).
3) Faz limpeza simples de valores faltantes (numéricos → 0; textuais → "-").
4) Aplica **subamostragem apenas nas classes dominantes** (configurável).
5) Realiza **split temporal 70/30 por classe** usando a coluna `ts`.
6) Remove `ts` e exporta `*_train.csv` e `*_test.csv`.

In [2]:
import re

# O caminho base é o diretório atual
base_path = '.'
pattern = '*_features_processed_timestamp.csv'

capture_dirs = sorted(glob.glob(os.path.join(base_path, 'captura_*_*')))
if not capture_dirs:
    raise RuntimeError("Nenhuma pasta captura_*_* encontrada no diretório atual.")

category_map = {
    "normal": "Normal",
    "reconnaissance": "Reconnaissance",
    "fuzzers": "Fuzzers",
    "analysis": "Analysis",
    "exploits": "Exploits",
    "dos": "DoS"
}

all_dataframes = []
loaded_files = []

print(f"Iniciando leitura dos arquivos '{pattern}'...")

for cap_dir in capture_dirs:
    folder = os.path.basename(cap_dir)
    m = re.match(r"^captura_(.+)_(\d+)$", folder)
    if not m:
        continue

    cat_raw = m.group(1).lower()
    section = int(m.group(2))
    attack_cat_from_folder = category_map.get(cat_raw, cat_raw.capitalize())

    files = sorted(glob.glob(os.path.join(cap_dir, pattern)))
    if not files:
        print(f"[AVISO] Nenhum '{pattern}' em {folder}")
        continue

    for file_path in files:
        try:
            print(f"Lendo: {file_path} ...")
            df_temp = pd.read_csv(file_path, low_memory=False)

            # metadados de seção (para split por seção)
            df_temp['capture_section'] = section
            df_temp['capture_folder'] = folder
            df_temp['capture_cat_from_folder'] = attack_cat_from_folder  # auditoria

            all_dataframes.append(df_temp)
            loaded_files.append(file_path)
        except Exception as e:
            print(f"Erro ao ler {file_path}: {e}")

print(f"Total de pastas captura_*_*: {len(capture_dirs)}")
print(f"Total de arquivos carregados: {len(loaded_files)}")


RuntimeError: Nenhuma pasta captura_*_* encontrada no diretório atual.

In [ ]:
# =========================
# 2) CONCATENAÇÃO + LIMPEZA
# =========================

if not all_dataframes:
    raise RuntimeError("Nenhum DataFrame foi carregado. Verifique os caminhos/pastas.")

final_dataset = pd.concat(all_dataframes, ignore_index=True)

# Normalizar nomes de colunas esperadas
rename_map = {
    "Label": "label",
    "duration": "dur",          # caso algum CSV tenha 'duration' em vez de 'dur'
    "conn_state": "state",      # caso exista essa variação
}
final_dataset = final_dataset.rename(columns={k:v for k,v in rename_map.items() if k in final_dataset.columns})

# Checagens mínimas
required_cols = ["attack_cat", "label"]
for c in required_cols:
    if c not in final_dataset.columns:
        raise KeyError(f"Coluna obrigatória ausente no dataset consolidado: '{c}'")

if "ts" not in final_dataset.columns:
    raise KeyError("Coluna 'ts' não encontrada. Ela é necessária para o split temporal.")

# Garantir tipos
final_dataset["label"] = pd.to_numeric(final_dataset["label"], errors="coerce").fillna(0).astype(int)
final_dataset["attack_cat"] = final_dataset["attack_cat"].astype(str)

# Limpeza: preenche NaN com base no tipo inferido
# - numéricos -> 0
# - textuais  -> "-"
num_cols = final_dataset.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = [c for c in final_dataset.columns if c not in num_cols]

final_dataset[num_cols] = final_dataset[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
final_dataset[obj_cols] = final_dataset[obj_cols].fillna("-")

# Garantir ts numérico (float) para ordenação
final_dataset["ts"] = pd.to_numeric(final_dataset["ts"], errors="coerce")
final_dataset = final_dataset.dropna(subset=["ts"]).copy()
final_dataset["ts"] = final_dataset["ts"].astype(float)

print("Dataset consolidado:")
print(" - linhas:", len(final_dataset))
print(" - colunas:", len(final_dataset.columns))
print("\nDistribuição (attack_cat):")
print(final_dataset["attack_cat"].value_counts())
# Auditoria: pasta vs attack_cat do arquivo (opcional)
if 'capture_cat_from_folder' in final_dataset.columns and 'attack_cat' in final_dataset.columns:
    mismatch = final_dataset[final_dataset['capture_cat_from_folder'] != final_dataset['attack_cat']]
    if len(mismatch) > 0:
        print(f"[AVISO] Há {len(mismatch)} linhas onde a pasta não bate com attack_cat do CSV. Verifique preprocessamento.")


In [ ]:
# =========================
# 3) SUBAMOSTRAGEM (APLICAR APENAS NO TREINO)
# =========================

SUBSAMPLE_FRAC = {
    "Exploits": 0.01,
    "DoS": 0.08,
    "Fuzzers": 0.3,
    "Reconnaissance": 0.14,
    "Analysis": 1
}

MIN_KEEP_PER_CLASS = 200  # garante um piso (ajuste conforme seu volume)

def subsample_by_fraction(df, cat, frac, min_keep=200, random_state=42):
    df_cat = df[df["attack_cat"] == cat]
    df_other = df[df["attack_cat"] != cat]
    n = len(df_cat)
    if n == 0:
        return df

    target = int(np.floor(n * float(frac)))
    target = max(target, min_keep)
    target = min(target, n)

    if target == n:
        return df

    df_cat_s = df_cat.sample(n=target, random_state=random_state)
    out = pd.concat([df_other, df_cat_s], ignore_index=True)
    return out

print("Subamostragem configurada. Ela será aplicada somente ao df_train após o split por seção.")


In [ ]:
# =========================
# 4) SPLIT 70/30 POR SEÇÃO (por classe)
# =========================
# Evita vazamento: nenhuma seção (captura_*_y) deve aparecer em treino e teste ao mesmo tempo.

TRAIN_RATIO = 0.70

train_parts = []
test_parts = []

for cat, df_cat in final_dataset.groupby("attack_cat", sort=False):
    sections = sorted(df_cat['capture_section'].unique())

    if len(sections) < 2:
        # Sem seções suficientes para dividir sem vazar
        train_parts.append(df_cat.copy())
        continue

    cut = int(np.floor(len(sections) * TRAIN_RATIO))
    cut = min(max(cut, 1), len(sections) - 1)

    train_secs = set(sections[:cut])
    test_secs  = set(sections[cut:])

    df_train_cat = df_cat[df_cat['capture_section'].isin(train_secs)].copy()
    df_test_cat  = df_cat[df_cat['capture_section'].isin(test_secs)].copy()

    if 'ts' in df_cat.columns:
        df_train_cat = df_train_cat.sort_values("ts").reset_index(drop=True)
        df_test_cat  = df_test_cat.sort_values("ts").reset_index(drop=True)

    train_parts.append(df_train_cat)
    test_parts.append(df_test_cat)

df_train = pd.concat(train_parts, ignore_index=True)
df_test  = pd.concat(test_parts, ignore_index=True)

# Subamostragem APENAS no treino (recomendado)
before_counts = df_train["attack_cat"].value_counts()
for cat, frac in SUBSAMPLE_FRAC.items():
    if cat in df_train["attack_cat"].unique():
        n0 = int(before_counts.get(cat, 0))
        df_train = subsample_by_fraction(df_train, cat, frac, min_keep=MIN_KEEP_PER_CLASS, random_state=RANDOM_STATE)
        n1 = int(df_train["attack_cat"].value_counts().get(cat, 0))
        print(f"Subamostragem treino '{cat}': {n0} -> {n1} (frac={frac})")

print("\nSplit concluído:")
print(" - treino:", len(df_train))
print(" - teste :", len(df_test))
print("\nDistribuição treino (attack_cat):")
print(df_train["attack_cat"].value_counts())
print("\nDistribuição teste (attack_cat):")
print(df_test["attack_cat"].value_counts())

# Checagem rápida de vazamento por seção
for cat in sorted(final_dataset["attack_cat"].unique()):
    tr = set(df_train.loc[df_train["attack_cat"]==cat, "capture_section"].unique())
    te = set(df_test.loc[df_test["attack_cat"]==cat, "capture_section"].unique())
    inter = tr & te
    if inter:
        print("[VAZAMENTO] ", cat, "seções em ambos:", sorted(inter))
